# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abood-arc/Flyrank-ml-project/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [4]:
print("""efore testing anything I looked at what these fields actually look like. This section pulls
from content with GSC data available in June twenty twenty six, no impressions floor yet, which
comes out to two hundred eight thousand six hundred thirty six items for impressions and clicks,
and two hundred one thousand eight hundred fifty three for position specifically, since that one
needs at least one day with a real position reading. Word count comes from the full content
library instead, five hundred nineteen thousand six hundred six items, since word count is just
a fact about the content itself and has nothing to do with any particular month.

Everything here is heavy tailed the way web traffic always is. Median clicks is literally zero,
more than half of all content gets no clicks at all in a month, but the mean is five point eight
and the max is a hundred fifty two thousand one hundred seventy. Impressions median is ninety
two, mean is a thousand thirty six, max is six hundred fifteen thousand. Position median is
twelve point nine but there is a page sitting at position five hundred seventy nine. Word count
is the least extreme of the four, median around twenty five hundred, but it is still missing
entirely for about a third of the library, thirty eight percent specifically for keyword
articles, which matches something already confirmed back in the ML-05 preview work.

None of this is a surprise on its own, but it is the reason every test after this uses weighted
sums or bucketed medians instead of a plain average. A plain mean gets dragged around by a
handful of giants in data shaped like this.
""")

efore testing anything I looked at what these fields actually look like. This section pulls
from content with GSC data available in June twenty twenty six, no impressions floor yet, which
comes out to two hundred eight thousand six hundred thirty six items for impressions and clicks,
and two hundred one thousand eight hundred fifty three for position specifically, since that one
needs at least one day with a real position reading. Word count comes from the full content
library instead, five hundred nineteen thousand six hundred six items, since word count is just
a fact about the content itself and has nothing to do with any particular month.

Everything here is heavy tailed the way web traffic always is. Median clicks is literally zero,
more than half of all content gets no clicks at all in a month, but the mean is five point eight
and the max is a hundred fifty two thousand one hundred seventy. Impressions median is ninety
two, mean is a thousand thirty six, max is six hundred fiftee

In [5]:

import os, getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute("SET enable_progress_bar=false")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
JUNE = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-06/data_0.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
QUERY_90D = f"read_parquet('{REL}/fact_content_query_90d.parquet')"

# impressions, clicks, position: any content with GSC data in June, no impressions floor yet.
# This is look-before-deciding, not the eligibility-floored population the later tests use.
dist_sql = f"""
    SELECT content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position
    FROM {JUNE}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
"""
dist = con.sql(dist_sql).df()

def describe(series, label):
    s = series.dropna()
    print(f'{label:12s} n={len(s):>7,}  min={s.min():>10,.1f}  p50={s.quantile(0.5):>10,.1f}  '
          f'p90={s.quantile(0.9):>10,.1f}  p99={s.quantile(0.99):>10,.1f}  max={s.max():>12,.1f}  mean={s.mean():>10,.1f}')

describe(dist['impressions'], 'impressions')
describe(dist['clicks'], 'clicks')
describe(dist['avg_position'], 'avg_position')

# word_count: static content fact, full library, no time window at all.
word_count = con.sql(f"SELECT word_count FROM {DIM_CONTENT}").df()
describe(word_count['word_count'], 'word_count')
print(f"word_count missing overall: {word_count['word_count'].isna().mean() * 100:.1f}%")

Paste your Hugging Face READ token (hf_...): ··········
impressions  n=208,636  min=       1.0  p50=      91.0  p90=   1,988.0  p99=  16,261.6  max=   615,012.0  mean=   1,036.2
clicks       n=208,636  min=       0.0  p50=       0.0  p90=       8.0  p99=      65.0  max=   152,170.0  mean=       5.8
avg_position n=201,853  min=       0.2  p50=      13.9  p90=      63.9  p99=      88.3  max=       579.0  mean=      24.2
word_count   n=341,838  min=       0.0  p50=   2,593.0  p90=   3,717.0  p99=   5,897.0  max=    29,341.0  mean=   2,472.1
word_count missing overall: 34.2%


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [6]:
print("""Three claims to check before I would trust any of them as real.

First, does longer content actually perform better. This one draws from June active content with
at least a hundred impressions that also has a recorded word count, eighty three thousand eight
hundred forty eight items after that word count requirement drops out the third of the library
where it is missing.

Second, are pages that rank for fewer distinct queries more fragile. This draws from the same
hundred impression floor but joined against the ninety day query table instead, ninety four
thousand one hundred fifty eight items, since not everything active in June has built up a full
ninety day query history yet.

Third, does GA4 engagement data add anything beyond GSC. This one just uses the hundred
impression floor on its own, no extra join, a hundred one thousand nine hundred seventeen items.
""")

Three claims to check before I would trust any of them as real.

First, does longer content actually perform better. This one draws from June active content with
at least a hundred impressions that also has a recorded word count, eighty three thousand eight
hundred forty eight items after that word count requirement drops out the third of the library
where it is missing.

Second, are pages that rank for fewer distinct queries more fragile. This draws from the same
hundred impression floor but joined against the ninety day query table instead, ninety four
thousand one hundred fifty eight items, since not everything active in June has built up a full
ninety day query history yet.

Third, does GA4 engagement data add anything beyond GSC. This one just uses the hundred
impression floor on its own, no extra join, a hundred one thousand nine hundred seventeen items.



In [7]:
# Signal 1: does longer content perform better?
# Population: June-active content (>=100 impressions) that also has a recorded word_count.
s1_sql = f"""
    WITH agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks
        FROM {JUNE} WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id HAVING SUM(gsc_impressions) >= 100
    ),
    joined AS (
        SELECT a.*, d.word_count FROM agg a JOIN {DIM_CONTENT} d USING (content_hash_id)
        WHERE d.word_count IS NOT NULL
    )
    SELECT
      CASE
        WHEN word_count < 1000 THEN '1_under_1000'
        WHEN word_count < 2000 THEN '2_1000-1999'
        WHEN word_count < 3000 THEN '3_2000-2999'
        WHEN word_count < 4000 THEN '4_3000-3999'
        ELSE '5_4000plus'
      END AS word_count_bucket,
      COUNT(*) AS n,
      SUM(impressions) AS total_impressions,
      SUM(clicks) AS total_clicks,
      ROUND(SUM(clicks)*100.0/SUM(impressions), 3) AS weighted_ctr_pct
    FROM joined GROUP BY 1 ORDER BY 1
"""
signal1 = con.sql(s1_sql).df()
print(f"n = {signal1['n'].sum():,}")
print(signal1.to_string(index=False))
print('VERDICT: MIXED. CTR peaks at 0.609% in the 2000-2999 word bucket, then drops to 0.443%')
print('for 3000-3999 before ticking back up. Not a monotonic longer-is-better story, more of a hump.')

# Signal 2: are pages ranking for fewer distinct queries more fragile?
# Population: June-active content (>=100 impressions) that also appears in the 90-day query table.
s2_sql = f"""
    WITH agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks
        FROM {JUNE} WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id HAVING SUM(gsc_impressions) >= 100
    ),
    per_content AS (
        SELECT content_hash_id, ANY_VALUE(content_visible_query_count) AS visible_queries
        FROM {QUERY_90D} GROUP BY content_hash_id
    ),
    joined AS (
        SELECT a.*, p.visible_queries FROM agg a JOIN per_content p USING (content_hash_id)
    )
    SELECT
      CASE
        WHEN visible_queries = 1 THEN '1_single_query'
        WHEN visible_queries <= 5 THEN '2_2to5'
        WHEN visible_queries <= 15 THEN '3_6to15'
        WHEN visible_queries <= 50 THEN '4_16to50'
        ELSE '5_51plus'
      END AS query_bucket,
      COUNT(*) AS n,
      SUM(impressions) AS total_impressions,
      SUM(clicks) AS total_clicks,
      ROUND(SUM(clicks)*100.0/SUM(impressions), 3) AS weighted_ctr_pct
    FROM joined GROUP BY 1 ORDER BY 1
"""
signal2 = con.sql(s2_sql).df()
print(f"n = {signal2['n'].sum():,}")
print(signal2.to_string(index=False))
print('VERDICT: OPPOSITE. My claim was fewer queries means more fragile, worse performance.')
print('CTR actually falls cleanly and monotonically as query count rises, 0.565% for single-query')
print('pages down to 0.326% for 51+ query pages. Precision beats breadth here, not the reverse.')

# Signal 3: does GA4 engagement data add real signal beyond GSC?
# Population: June-active content (>=100 impressions), no further filtering.
s3_sql = f"""
    WITH agg AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks,
               BOOL_OR(ga4_data_available) AS ga4_avail
        FROM {JUNE} WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id HAVING SUM(gsc_impressions) >= 100
    )
    SELECT ga4_avail, COUNT(*) AS n, ROUND(SUM(clicks)*100.0/SUM(impressions),3) AS weighted_ctr_pct
    FROM agg GROUP BY 1
"""
signal3 = con.sql(s3_sql).df()
print(f"n = {signal3['n'].sum():,}")
print(signal3.to_string(index=False))

# Row-level coverage check, since item-level BOOL_OR can mark a page "available" off just one
# good day out of thirty. This is the honest coverage number, matching how ML-04 checked it.
row_level_sql = f"""
    SELECT ga4_data_available, COUNT(*) AS n_rows,
           ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 2) AS pct_of_rows
    FROM {JUNE} GROUP BY 1
"""
row_level = con.sql(row_level_sql).df()
print(row_level.to_string(index=False))
print('VERDICT: MIXED. Row-level GA4 coverage is only about 5.5% in June, consistent with the')
print('4.2% already documented for March in the ML-04 findings, so this stays a thin signal.')
print('Where GA4 data does show up, CTR is higher, but that is very likely explained by which')
print('clients happened to adopt GA4 rather than GA4 itself causing anything.')


n = 83,848
word_count_bucket     n  total_impressions  total_clicks  weighted_ctr_pct
     1_under_1000   113           950015.0        4319.0             0.455
      2_1000-1999  4040         12838644.0       59189.0             0.461
      3_2000-2999 52365        127251025.0      774826.0             0.609
      4_3000-3999 21485         44226301.0      196066.0             0.443
       5_4000plus  5845          5994094.0       29793.0             0.497
VERDICT: MIXED. CTR peaks at 0.609% in the 2000-2999 word bucket, then drops to 0.443%
for 3000-3999 before ticking back up. Not a monotonic longer-is-better story, more of a hump.
n = 94,158
  query_bucket     n  total_impressions  total_clicks  weighted_ctr_pct
1_single_query  6635          2234690.0       12620.0             0.565
        2_2to5 21756          9829799.0       52799.0             0.537
       3_6to15 29664         24019285.0      115016.0             0.479
      4_16to50 25154         56434024.0      234595.0      

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [8]:

print("""Now the one tied to a real FlyRank flag. Quick win logic assumes fixing a high volume page
matters more than fixing a low volume one, which only makes sense if volume is actually
concentrated rather than spread evenly. Same population as the GA4 check above, the plain
hundred impression floor, a hundred one thousand nine hundred seventeen items, split into ten
equal sized groups by volume.""")

Now the one tied to a real FlyRank flag. Quick win logic assumes fixing a high volume page
matters more than fixing a low volume one, which only makes sense if volume is actually
concentrated rather than spread evenly. Same population as the GA4 check above, the plain
hundred impression floor, a hundred one thousand nine hundred seventeen items, split into ten
equal sized groups by volume.


In [9]:
# Flag-linked signal: volume, behind the quick-win flag.
# Claim: fixing a high volume page matters more, which only holds if volume is genuinely
# concentrated rather than spread roughly evenly across content.
# Population: same as signal 3, June-active content (>=100 impressions), no further filtering.
flag_sql = f"""
    WITH agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions
        FROM {JUNE} WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id HAVING SUM(gsc_impressions) >= 100
    ),
    ranked AS (
        SELECT *, NTILE(10) OVER (ORDER BY impressions DESC) AS decile
        FROM agg
    )
    SELECT decile, COUNT(*) AS n, SUM(impressions) AS total_impressions,
           ROUND(100.0*SUM(impressions) / SUM(SUM(impressions)) OVER (), 2) AS pct_of_all_impressions
    FROM ranked GROUP BY 1 ORDER BY 1
"""
flag_test = con.sql(flag_sql).df()
print(f"n = {flag_test['n'].sum():,}")
print(flag_test.to_string(index=False))
print('VERDICT: CONFIRMED. The top decile by volume alone holds 64.3% of all impressions.')
print('Volume-based prioritization is capturing where the real stake sits, not sorting on noise.')



n = 101,917
 decile     n  total_impressions  pct_of_all_impressions
      1 10192        137472916.0                   64.33
      2 10192         30469478.0                   14.26
      3 10192         16084239.0                    7.53
      4 10192          9968024.0                    4.66
      5 10192          6608659.0                    3.09
      6 10192          4573317.0                    2.14
      7 10192          3253559.0                    1.52
      8 10191          2349398.0                    1.10
      9 10191          1697562.0                    0.79
     10 10191          1211846.0                    0.57
VERDICT: CONFIRMED. The top decile by volume alone holds 64.3% of all impressions.
Volume-based prioritization is capturing where the real stake sits, not sorting on noise.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [10]:
print("""Write length on its own is not a lever. The sweet spot sits around two to three thousand
words and pushing past that does not reliably help, so word count alone is not a proxy for
quality. A page ranking for one tightly matched query is doing better than one spread thin
across fifty, so breadth is not automatically a strength, focus is. GA4 is still too sparse to
lean on across the board, keep it as a bonus signal for the clients who have it rather than a
co-equal one. And volume really is where the leverage is, prioritizing fixes by traffic size is
not a lazy shortcut, the data backs it directly.""")

Write length on its own is not a lever. The sweet spot sits around two to three thousand
words and pushing past that does not reliably help, so word count alone is not a proxy for
quality. A page ranking for one tightly matched query is doing better than one spread thin
across fifty, so breadth is not automatically a strength, focus is. GA4 is still too sparse to
lean on across the board, keep it as a bonus signal for the clients who have it rather than a
co-equal one. And volume really is where the leverage is, prioritizing fixes by traffic size is
not a lazy shortcut, the data backs it directly.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.